# Raman protein perturbation pub

In [ ]:
from raman_analysis.utils import collect_spectra_from_dir, plot_spectra_in_dir, add_peak_annotations
import plotly.express as px
import ramanspy as rp
import pandas as pd
import plotly.colors as pcolors
import numpy as np
from pathlib import Path
import arcadia_pycolor as apc
apc.plotly.setup()
%load_ext autoreload
%autoreload 2

real_NAS = Path('/Volumes/Microscopy')
NAS_mirror = Path('/Users/ilya/NAS_mirror')
raman_chem_perturbation_dir = Path('data/raman-chemical-perturbations')
raman_ph_perturbation_dir = Path('data/raman-ph-perturbations')
plot_dir_htmls = Path('plots/html')
plot_dir_pdf = Path('plots/pdf')

plot_scale = 3
fig_width = 1000
fig_height=400

In [ ]:
apc.palettes.primary

In [ ]:
def generate_subtract_dict(samples):
    """ Generate a dictionary mapping sample names to their buffer names for subtraction. """
    subtract_dict = {}
    for name in samples:
        date = name.split('/')[0]
        buffer = name.split(' in ')[-1]
        subtract_dict[name] = f"{date}/{buffer}"
    return subtract_dict

# Playground (will delete)

In [ ]:
# data_dir = raman_chem_perturbation_dir

# data_list, sample_df = collect_spectra_from_dir(data_dir, 'instantraman',recursive=True)

# pipeline = rp.preprocessing.Pipeline(
#     [
#         rp.preprocessing.misc.Cropper(region=(350, 2000)),
#         # rp.preprocessing.despike.WhitakerHayes(),
#         rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
#         # rp.preprocessing.baseline.ASPLS(lam=1e5),
#         rp.preprocessing.baseline.ModPoly(poly_order=4),
#         # rp.preprocessing.normalise.AUC(),
#     ]
# )

# n_groups = sample_df['Group ID'].nunique()

# include_samples = [n for n in sample_df['Name'] if "Hemoglobin [low]" in n]
# # include_samples = [n for n in sample_df['Name'] if any(x in n for x in [" S4", " S5"])]
# # include_samples = [n for n in sample_df['Name'] if any(x in n for x in [" S1", " S2", " S3"])]

# sampled_custom_colors = pcolors.sample_colorscale(px.colors.qualitative.Dark2, samplepoints=n_groups)

# subtract_id = generate_subtract_dict(include_samples)

# f, names, rp_container = plot_spectra_in_dir(data_list,
#                     col_to_plot = "Intensity (dark-subtracted)",
#                     data_dir = data_dir,
#                     include=None,
#                     rp_pipeline=pipeline,
#                     colors=sampled_custom_colors,
#                     height=600,
#                     width=1000,
#                     subtract_id=subtract_id,# '2025-12-10/Tris',
#                     # y_offset=0.002,
#                     xaxis_title='Wavenumber (cm⁻¹)',
#                     yaxis_title='Intensity (a.u.)',
# )
# # f.write_html("plots/2025-10-31-buffers.html")
# f.show()


# Protein perturbations

* Data from 2025-12-{10,13}

In [ ]:
data_dir = raman_chem_perturbation_dir

data_list, sample_df = collect_spectra_from_dir(data_dir, 'instantraman', recursive=True)

pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ASPLS(lam=5e4),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
        # rp.preprocessing.normalise.AUC(),
    ]
)


## All plottable samples

In [ ]:
buffers = ['Tris', 'Tris+TCEP', 'Tris+EDTA', 'Tris+TCEP+EDTA']
all_samples = sample_df['Name'].str.split('/').str[-1].str.split(' in ').str[0].unique()
all_samples = [s for s in all_samples if not any(b in s for b in buffers)]
all_samples

In [ ]:
xranges = {'BSA(10mgml)': [[None,1500], [360, 700], [1150,1378]],
           'DNAse': [None, [360, 700], [1150,1378]],
           'CytC [high]': [None, [360, 700], [1500,1650]]
           }
yranges = {'BSA(10mgml)': [None, [-700,1000], [None, 2500]],
           'DNAse': [None, [-300, 1500], None],
           'CytC [high]': [None, None, None]}

fig_heights = {'BSA(10mgml)': [fig_height, fig_height, fig_height],
               'DNAse': [fig_height, fig_height, fig_height],
               'CytC [high]': [fig_height, fig_height, fig_height]}
fig_widths = {'BSA(10mgml)': [fig_width, fig_width*0.5, fig_width*0.5],
              'DNAse': [fig_width, fig_width*0.5, fig_width*0.5],
              'CytC [high]': [fig_width, fig_width*0.5, fig_width*0.5]}

n_groups = sample_df['Group ID'].nunique()

sampled_custom_colors = apc.palettes.primary # pcolors.sample_colorscale(px.colors.qualitative.Bold, samplepoints=n_groups)


buffer_dict = {'BSA(10mgml)': buffers[:-1], # exclude TCEP+EDTA condition for BSA since it causes precipitation
           'DNAse': buffers,
           'CytC [high]': buffers}


for protein in xranges.keys():
    for i, (xrange, yrange) in enumerate(zip(xranges[protein], yranges[protein])):
        include_samples = [n for n in sample_df['Name'] if (any(n.endswith(buf) for buf in buffer_dict[protein]) and protein in n)]
        subtract_id = generate_subtract_dict(include_samples)
        include_samples += list(subtract_id.values())
        f, names, rp_container = plot_spectra_in_dir(data_list,
                            col_to_plot = "Intensity (dark-subtracted)",
                            data_dir = data_dir,
                            subtract_id=subtract_id, # None if ('BSA' in protein) and ('Tris+TCEP+EDTA' in buffer_subset) else subtract_id,
                            rp_pipeline=pipeline,
                            colors=sampled_custom_colors,
                            include=include_samples,
                            height=fig_heights[protein][i],
                            width=fig_widths[protein][i],
                            xaxis_title='Wavenumber (cm⁻¹)',
                            yaxis_title='Intensity (a.u.)',
                            title=f"{protein} | {xrange}_cm-1 | {'_'.join(buffer_dict[protein])}"
                            )
        f.update_xaxes(range=xrange)
        f.update_yaxes(range=yrange)
        # f.data = tuple(trace for trace in f.data if trace.name != subtract_id)
        f.data = tuple(trace for trace in f.data if trace.name not in subtract_id.values())
        f.update_layout(showlegend=False)
        apc.plotly.style_plot(f, monospaced_axes="all")

        
        # protein_dir_pdf = plot_dir_pdf / protein
        # protein_dir_html = plot_dir_htmls / protein
        # protein_dir_pdf.mkdir(parents=True, exist_ok=True)
        # protein_dir_html.mkdir(parents=True, exist_ok=True)
        
        # f.write_image(protein_dir_pdf / f"{protein}_{xrange}_cm-1.pdf", scale=plot_scale)
        # f.write_html(protein_dir_html / f"{protein}_{xrange}_cm-1.html")
        f.show()



## Buffers alone

In [ ]:
buffers

In [ ]:
include_samples = [n for n in sample_df['Name'] if (any(n == f'2025-12-10/{buf}' for buf in buffers))]

f, names, rp_container = plot_spectra_in_dir(data_list,
                    col_to_plot = "Intensity (dark-subtracted)",
                    data_dir = data_dir,
                    subtract_id=None,
                    rp_pipeline=pipeline,
                    colors=sampled_custom_colors,
                    include=include_samples,
                    height=fig_height,
                    width=fig_width,
                    # y_offset=0.002,
                    xaxis_title='Wavenumber (cm⁻¹)',
                    yaxis_title='Intensity (a.u.)',
                    title="Buffers",
                    )

f.update_layout(showlegend=False)
apc.plotly.style_plot(f, monospaced_axes="all")
f.write_image(plot_dir_pdf / "buffers.pdf", scale=plot_scale)
f.write_html(plot_dir_htmls / "buffers.html")
f.show()

# pH perturbations

In [ ]:
data_list, sample_df = collect_spectra_from_dir(raman_ph_perturbation_dir, 'instantraman', recursive=True)

pipeline = rp.preprocessing.Pipeline(
    [
        rp.preprocessing.misc.Cropper(region=(350, 1800)),
        rp.preprocessing.despike.WhitakerHayes(),
        rp.preprocessing.denoise.SavGol(window_length=5, polyorder=3),
        # rp.preprocessing.baseline.ASPLS(lam=5e4),
        rp.preprocessing.baseline.ModPoly(poly_order=9),
        # rp.preprocessing.normalise.AUC(),
    ]
)

n_groups = sample_df['Group ID'].nunique()


In [ ]:
def generate_subtract_dict_pH(_replicate, sample_string):
    ph_vals = [0, 0.3, 1, 1.3, 2, 2.3, 3, 3.3, 4, 5, 6, 7.5, 8, 9, 9.5, 10, 10.7, 11, 11.7, 12, 12.7, 13, 13.7, 14]
    ph_vals_str = [str(p) for p in ph_vals]

    sample_names = sample_df['Name'].unique().tolist()
    all_protein = [n for n in sample_names if sample_string + " pH" in n and _replicate in n]

    subtraction_key = 'TCEP' if 'TCEP' in sample_string else 'buffers'
    subtract_dict = {}
    for p in all_protein:
        for ph in ph_vals_str:
            if f"{sample_string} pH {ph}" in p:
                subtract_dict[p] = [b for b in sample_names if b.endswith(f"{_replicate}/{subtraction_key} pH {ph}")][0]
    return subtract_dict


In [ ]:
ph_gradient = apc.Gradient("ph_gradient", [apc.cinnabar, apc.chateau, apc.lapis])
BSA_color = apc.amber
BSA_TCEP_color = apc.vital


In [ ]:
# ph_to_plot = [1, 4, 9, 12, 14]
ph_to_plot = [0.3, 1, 1.3, 2, 2.3, 3, 3.3, 4, 5, 6, 7.5, 8, 9, 9.5, 10, 10.7, 11, 11.7, 12, 12.7, 13]
replicate = 'rep1'
ph_to_plot_labels = [f"pH {ph_val}" for ph_val in ph_to_plot]


sample_names_dict = {}
sample_containers_dict = {}

for sample in ['BSA', 'BSA+TCEP']:
    subtract_dict = generate_subtract_dict_pH(replicate, sample)
    subtract_dict = {k: v for k, v in subtract_dict.items() if any(k.endswith(label) for label in ph_to_plot_labels)}

    include_samples = list(subtract_dict.keys()) + list(subtract_dict.values())
    exclude_from_plot = subtract_dict.values()

    colors = pcolors.sample_colorscale(px.colors.qualitative.Dark24, samplepoints=n_groups)

    f, names, rp_container = plot_spectra_in_dir(data_list,
                        col_to_plot = "Intensity (dark-subtracted)",
                        data_dir = raman_ph_perturbation_dir,
                        subtract_id=subtract_dict,
                        rp_pipeline=pipeline,
                        colors=colors,
                        include=include_samples,
                        height=fig_height,
                        width=fig_width,
                        xaxis_title='Wavenumber (cm⁻¹)',
                        yaxis_title='Intensity (a.u.)',
                        title=f"{sample} pH perturbation | {replicate}"
    )
    f.data = tuple(trace for trace in f.data if not any(trace.name == exclude for exclude in exclude_from_plot))
    apc.plotly.style_plot(f, monospaced_axes="all")

    gradient = ph_gradient.resample_as_palette(len(ph_to_plot_labels))
    for i, trace in enumerate(f.data):
        trace.line.color = gradient[i]
    f.show()

    sample_names_dict[sample] = names
    sample_containers_dict[sample] = rp_container


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def intensity_at(spectral_axis, spectral_data_row, wavenumber):
    idx = np.argmin(np.abs(spectral_axis - wavenumber))
    return float(spectral_data_row[idx])

peak_records = []
for sample in ['BSA', 'BSA+TCEP']:
    names = sample_names_dict[sample]
    container = sample_containers_dict[sample]
    ax = container.spectral_axis
    for i, name in enumerate(names):
        if 'BSA' not in name:
            continue
        for ph in ph_to_plot:
            if not name.endswith(f"pH {ph}"):
                continue
            row = container.spectral_data[i]
            peak_records.append({
                'pH': ph,
                'sample': sample,
                'i515': intensity_at(ax, row, 515),
                'ratio_830_854': intensity_at(ax, row, 830) / intensity_at(ax, row, 854),
                'ratio_1343_1256': intensity_at(ax, row, 1343) / intensity_at(ax, row, 1256),
                'i1657': intensity_at(ax, row, 1657),
            })

peak_df = pd.DataFrame(peak_records)

peak_params = [
    ('i515',           'Peak amplitude at 515 cm⁻¹'),
    ('ratio_830_854',  '830 cm⁻¹ / 854 cm⁻¹'),
    ('ratio_1343_1256','1343 cm⁻¹ / 1256 cm⁻¹'),
    ('i1657',          'Peak amplitude at 1657 cm⁻¹'),
]

sample_colors = {'BSA': BSA_color, 'BSA+TCEP': BSA_TCEP_color}

fig = make_subplots(rows=2, cols=2, subplot_titles=[label for _, label in peak_params],
                    vertical_spacing=0.4)
for idx, (col, label) in enumerate(peak_params):
    row_idx, col_idx = divmod(idx, 2)
    for sample in ['BSA', 'BSA+TCEP']:
        df_s = peak_df[peak_df['sample'] == sample].sort_values('pH')
        fig.add_trace(
            go.Scatter(
                x=df_s['pH'], y=df_s[col],
                mode='lines+markers',
                name=sample,
                legendgroup=sample,
                showlegend=(idx == 0),
                line=dict(color=sample_colors[sample]),
            ),
            row=row_idx + 1, col=col_idx + 1,
        )
fig.update_layout(
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1
    )
)
fig.update_xaxes(title_text='pH')
apc.plotly.style_plot(fig, monospaced_axes="all")
apc.plotly.set_figure_dimensions(fig, size="float")
fig.show()
